# 第11课 · 向量（vector）范数 — L1 / L2 / ∞ 范数的计算、几何形状与正则化含义

**目标**：会算向量长度(L2 范数)、向量间距离，会把向量归一化（normalization）成单位长度。

**为什么对 Aurora 重要**：MSE 损失就是误差向量的 L2 范数平方；L1/L2 正则项直接约束权重向量的范数；MFCC 特征送入模型前先按 L2 范数归一化，让不同帧的能量落在同一尺度。

← **上一课**　[L10 · 点积与投影](L10_dot_product.ipynb)

> 上节课学习了 **点积与投影**：a·b = |a||b|cosθ，为什么相似度 = 点积 ÷ 积范数。  
> 本课将探讨 **向量范数**。

## 本课剧情：量"距离"有多种尺子

你在地图上量两点之间的距离，有几种方法：
- **直线距离**（鸟飞过去的路）：用勾股定理，这就是 **L2 范数**
- **街区距离**（出租车沿街道走的路）：沿x轴走+沿y轴走，这就是 **L1 范数**
- **最远的那步**（棋盘上国王走法）：只看最长的那个分量，这就是 **L∞ 范数**

三把"尺子"，三种不同的"长度"定义。哪种合适，取决于任务：
- **L2**：几何距离，MFCC 归一化，神经网络权重衰减（L2 regularization）
- **L1**：稀疏性，Lasso 回归，对异常值更鲁棒
- **L∞**：最坏情况下的误差界

本课唯一的练习：实现 `normalize(v)`——把任意向量压缩到单位 L2 球上（长度变为 1）。

## 1. 三种范数，三把尺子

给向量 `v = [v₁, v₂, ..., vₙ]`，三种"长度"定义：

| 范数 | 公式 | 几何含义 | Aurora 用途 |
|---|---|---|---|
| L2（欧式）| `√(Σ vᵢ²)` | 直线距离（勾股定理推广）| MFCC 归一化、余弦相似度分母 |
| L1（曼哈顿）| `Σ |vᵢ|` | 街区距离 | Lasso 正则化，对异常值鲁棒 |
| L∞（切比雪夫）| `max(|vᵢ|)` | 最大分量 | 最坏误差界 |

**手算练习**（v = [3, 4]，先在纸上算，再运行代码对答案）：

| 范数 | 手算 |
|---|---|
| L2 | ✏️ √(9+16) = ? |
| L1 | ✏️ 3+4 = ? |
| L∞ | ✏️ max(3,4) = ? |

## 符号入口：先看范数，再看归一化

L2 范数写作 `‖v‖₂`，L1 写作 `‖v‖₁`。给向量 `v = [3, 4]`：`‖v‖₂ = sqrt(9+16) = 5`，`‖v‖₁ = 3+4 = 7`。两个范数都量「长度」，但 L2 对大分量惩罚更重，L1 对各分量一视同仁。

In [ ]:
import numpy as np
v = np.array([3.0, 4.0])
print('手算:', np.sqrt(np.sum(v**2)))   # 5.0
print('numpy:', np.linalg.norm(v))      # 5.0
print('L1 范数(绝对值之和):', np.sum(np.abs(v)))

## 动手观察：同一向量，两种长度

运行下方代码，注意 L1 和 L2 给出的数值不同，但归一化后向量的 L2 范数恰好等于 1。改动某个分量的大小，观察哪个范数变化更剧烈。

In [ ]:
import numpy as np

# L1 vs L2 vs L∞ 范数的数值对比
vecs = [np.array([3., 4.]), np.array([1., 0., 0., 10.]), np.array([-2., 2., -2.])]
for v in vecs:
    l1 = np.sum(np.abs(v))
    l2 = np.sqrt(np.sum(v**2))
    linf = np.max(np.abs(v))
    print(f'v={v}  L1={l1:.2f}  L2={l2:.2f}  L∞={linf:.2f}')
print('观察：L2 比 L1 小（三角不等式），L∞ 最小（只看最大分量）。')


## 代码实验：遍历几个向量，对比 L1 / L2 范数

L2 范数对分量中的极端值更敏感——一个很大的分量会显著拉高 L2、但对 L1 的影响只是线性的。下面的循环展示这个差距如何随向量的「尖锐程度」变化。

In [ ]:
import numpy as np

# 对离群值的敏感度：L1 比 L2 更鲁棒
base = np.ones(10)
for outlier in [0, 1, 5, 10, 50]:
    v = base.copy(); v[0] = outlier
    print(f'v[0]={outlier:>3d}  L1={np.linalg.norm(v,1):.1f}  '
          f'L2={np.linalg.norm(v,2):.2f}  L∞={np.linalg.norm(v,np.inf):.0f}')
print('→ L2 被离群值放大（平方），L1 线性增长，L∞ 直接等于离群值。')


## 2. 两个向量的距离 = 差向量的长度

In [ ]:
a = np.array([1.0, 2.0]); b = np.array([4.0, 6.0])
print('距离 =', np.linalg.norm(a - b))  # 5.0

## 3. ✏️ 你的任务：实现 `normalize`（缩放到长度为 1）

**推理路线**：
1. 输入 `v` 是形状 `(n,)` 的一维数组，目标是让输出向量的 L2 范数等于 1。
2. 先算 `length = np.linalg.norm(v)`——向量的「总长度」。
3. 用 `v / length` 把每个分量同比缩小：各分量比值（方向）不变，长度从 `length` 变成 `length/length = 1`。关键点：不能用 `v / np.sum(v)` 或 `v / np.max(v)`，那些不能保证 `‖result‖₂ = 1`。
4. **零向量边界**：若 `length == 0`，`v / length` 产生 NaN。约定：零向量应返回全零向量。用 eps 保护：`v / (np.linalg.norm(v) + 1e-8)`——分子全零，结果仍约为零向量。

**参考输入输出**：`v=[3,4]` → 范数 5，归一化后 `[0.6, 0.8]`，验证 `0.6²+0.8²=1.0`

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


### 写 `normalize` 前明确三件事

- 输入：`v`，形状 `(n,)` 的浮点向量
- 关键步骤：计算 `length = np.linalg.norm(v)`，再做 `v / length`
- 返回：与 `v` 形状相同、L2 范数等于 1 的单位向量

In [ ]:
def normalize(v):
    # ✏️ TODO: 返回单位长度向量（提示：v / (np.linalg.norm(v) + 1e-8)，零向量结果≈全零）
    raise NotImplementedError("TODO: divide v by its L2 norm")


In [ ]:
try:
    u = normalize(np.array([3.0, 4.0]))
    print("归一化后:", u, "| 长度:", round(float(np.linalg.norm(u)), 6))
    assert abs(np.linalg.norm(u) - 1.0) < 1e-9, "归一化后长度应为 1"
    assert np.allclose(u, [0.6, 0.8], atol=1e-9), "方向错误：v=[3,4] 应归一化为 [0.6, 0.8]"

    # 负分量：方向不受符号影响
    u_neg = normalize(np.array([-6.0, 8.0]))
    assert abs(np.linalg.norm(u_neg) - 1.0) < 1e-9, "负分量向量归一化后长度仍应为 1"
    assert np.allclose(u_neg, [-0.6, 0.8], atol=1e-9), "方向错误：v=[-6,8] 应归一化为 [-0.6, 0.8]"

    # 零向量：eps 保护，不产生 nan，不抛出异常
    z = normalize(np.array([0.0, 0.0]))
    assert not np.any(np.isnan(z)), "零向量归一化不应产生 nan"
    print("\n✅ 通过：[3,4] 方向、负分量方向、零向量三种边界全过。")
except NotImplementedError as e:
    print(f"⚠️  尚未实现，请完成 normalize()：{e}")


**🔗 Aurora 连接**：`normalize(v)` 在 Aurora 的 MFCC 流水线中对每帧特征调用，归一化后余弦相似度（cosine similarity）等价于 `np.dot(a, b)`。说话人识别比对的是 d-vector 方向而非幅度——两段录音音量差异很大，归一化后方向相近则判定为同一说话人。

## 🎨 图示：L2 范数 = 箭头长度 (3,4)→5

In [ ]:
from aurora.laviz import style, arrows2d
style()
arrows2d([[3,4]], ['v, |v|=5'], title='范数 = 长度');

In [ ]:
import numpy as np

# 不同 p 下，向量 [a, 1-a] 在 Lp 单位球上的 a 值（1D 截面）
# Lp 球：|x1|^p + |x2|^p = 1 => x2 = (1 - |x1|^p)^{1/p}
x1 = 0.6
for p in [1, 2, 4, np.inf]:
    if np.isinf(p):
        x2 = 1.0   # L∞ 球是正方形，点 (0.6, 1.0) 在上边界上（真正角点为 (±1, ±1)）
    else:
        val = max(0.0, 1 - abs(x1)**p)
        x2 = val**(1/p)
    lbl = '∞' if np.isinf(p) else str(int(p))
    print(f'L{lbl} 球上 x1={x1}: x2={x2:.4f}  (范数验证={np.linalg.norm([x1,x2],p if not np.isinf(p) else np.inf):.4f})')


## 参数实验：只改一个旋钮

把 `v=[3, 4]` 缩放 10 倍变成 `[30, 40]`，对比归一化结果——应仍为 `[0.6, 0.8]`，方向不随幅度改变。再把全零向量 `[0, 0]` 传入 `normalize`，观察 `nan` 或 `ZeroDivisionError`——实际代码里需要加 `eps` 保护：`v / (np.linalg.norm(v) + 1e-8)`。

## 4. 正则化含义：L1 vs L2 梯度差异

L1 正则化（Lasso）在损失函数中加 `λ‖w‖₁`，对参数 `w` 的梯度为 `λ·sign(w)`——无论 `w` 多小，梯度绝对值恒为 λ，会把小权重直接推向零（**稀疏性**）。

L2 正则化（Ridge / 权重衰减）加 `λ‖w‖₂²`，梯度为 `2λw`——权重越小梯度越小，只做等比例收缩，不会精确归零（**shrinkage**）。

直觉对比：L1 对所有权重施加固定大小的"拉力"；L2 施加与权重大小成正比的"弹力"。


In [ ]:
import numpy as np

# L1 vs L2 正则化的梯度对比（梯度下降中的权重更新方向）
weights = np.array([2.0, 0.5, 0.05, 0.001])   # 四个大小不同的权重
lambda_ = 0.1

print("权重 w      L1 梯度 λ·sign(w)    L2 梯度 2λ·w")
print('-' * 52)
for w in weights:
    grad_l1 = lambda_ * np.sign(w)     # 恒为 ±λ，与 w 大小无关
    grad_l2 = 2 * lambda_ * w          # 正比于 w
    print(f"  w={w:.3f}       grad_L1={grad_l1:+.4f}         grad_L2={grad_l2:+.6f}")

print()
print("结论：")
print("  L1 梯度恒为 ±λ，无论权重多小都受同等力推向零  → 稀疏（许多权重精确归零）")
print("  L2 梯度正比于 w，小权重受力更弱                → 仅收缩，不归零（权重衰减）")


## 本课收束

现在可以用 `normalize(v)` 把任意非零向量压到单位球上，L2 长度由 `np.linalg.norm(v)` 计算。Aurora 的 MFCC 流水线在每帧特征送入分类器前调用 `normalize`，让不同录音环境下的能量值落在同一尺度。MSE 损失函数 `mean((y_pred - y_true)**2)` 等价于对误差向量取 L2 范数再平方再平均。下一课：**L12** 的矩阵乘法中，矩阵每列的 L2 范数决定它把输入向量拉伸多少倍。

---
⬇️ **通关检验**：收束小结已读；请完成下方白板挑战后再勾选自评。


## ✏️ 白板挑战：范数手算（目标 8 分钟）

盖上屏幕，纸上作答：

**向量**：v = [3, 4]，u = [1, 0, 0]，w = [-1, 2, -3]

**问 1**：v 的 L2 范数是多少？L1 范数？L∞ 范数？

**问 2**：normalize(v) 的结果是什么？验证它的 L2 范数 = 1。
公式：normalize(v) = v / ‖v‖₂

**问 3**：两向量 a = [1, 2]，b = [4, 6] 之间的 L2 距离是多少？
公式：‖a - b‖₂ = √((4-1)² + (6-2)²) = ?

**问 4**：向量 w = [-1, 2, -3] 的 L∞ 范数是多少？

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import numpy as np

v = np.array([3.0, 4.0])

# 问1：三种范数
assert np.isclose(np.linalg.norm(v, 2), 5.0, atol=1e-10)
assert np.isclose(np.linalg.norm(v, 1), 7.0, atol=1e-10)
assert np.isclose(np.linalg.norm(v, np.inf), 4.0, atol=1e-10)
print(f"Q1 ✅  v=[3,4]: L2={np.linalg.norm(v):.1f}, L1={np.linalg.norm(v,1):.1f}, L∞={np.linalg.norm(v,np.inf):.1f}")

# 问2：归一化
try:
    nv = normalize(v)
    assert np.isclose(np.linalg.norm(nv), 1.0, atol=1e-9)
    print(f"Q2 ✅  normalize([3,4]) = {np.round(nv,4)}  ‖‖₂ = {np.linalg.norm(nv):.6f}")
except NotImplementedError:
    print("⬜ Q2：请先实现 normalize()，再运行对答案格")

# 问3：L2 距离
a, b = np.array([1.0, 2.0]), np.array([4.0, 6.0])
dist = np.linalg.norm(a - b)
assert np.isclose(dist, 5.0, atol=1e-10)  # √(9+16) = 5
print(f"Q3 ✅  ‖[1,2]-[4,6]‖₂ = √(9+16) = {dist:.1f}")

# 问4：L∞ 范数
w = np.array([-1.0, 2.0, -3.0])
linf = np.linalg.norm(w, np.inf)
assert np.isclose(linf, 3.0, atol=1e-10)
print(f"Q4 ✅  ‖[-1,2,-3]‖∞ = max(1,2,3) = {linf:.1f}")
print("\n🎉 范数白板挑战通过！三把'尺子'已内化。")

In [ ]:
# ✏️ 本课自评
l11_review = {
    "normalize_implemented":  None,  # normalize 实现并通过断言？True/False
    "three_norms_understood": None,  # 能说出 L1/L2/L∞ 的公式和几何含义？True/False
    "regularization_intuition": None, # 理解 L1→稀疏、L2→权重衰减？True/False
    "whiteboard_passed":      None,  # 白板挑战纸上推导完成？True/False
}

unfilled = [k for k, v in l11_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l11_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L11 全部通关！进入 L12：矩阵乘法')

---

→ **下一课**　[L12 · 矩阵乘法](L12_matrices.ipynb)

> 下节课将学习 **矩阵乘法**：矩阵 = 线性变换，乘法 = 函数复合，手推 2×2 例子。